In [4]:
import torch
import torch.nn as nn

class block(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        """
        A single Residual Block in the ResNet model.
        Args:
            in_channels: Number of input channels.
            out_channels: Number of output channels.
            stride: Stride for the convolutional layer.
            downsample: Downsampling layer to match dimensions for the skip connection.
        """
        super(block, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn


In [5]:
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=4, input_size=(3, 224, 224)):
        """
        ResNet model using Residual Blocks.
        Args:
            block: Residual block class.
            layers: List defining the number of blocks in each layer.
            num_classes: Number of output classes for classification.
            input_size: Tuple representing the input size (C, H, W).
        """
        super(ResNet, self).__init__()
        self.in_channels = 64

        # Initial convolutional layer
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet layers
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        # Dynamically calculate the flattened size
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Fully connected layer
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride=1):
        """
        Creates a ResNet layer with the specified number of blocks.
        Args:
            block: Residual block class.
            out_channels: Number of output channels.
            blocks: Number of blocks in the layer.
            stride: Stride for the first block.
        """
        downsample = None
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

        layers = [block(self.in_channels, out_channels, stride, downsample)]
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(block(out_channels, out_channels))

        return nn.Sequential(*layers)

    def _get_flattened_size(self, input_size):
        """
        Calculate the size of the flattened tensor after passing through the feature extractor.
        Args:
            input_size: Tuple representing the input size (C, H, W).
        Returns:
            int: Flattened size of the tensor.
        """
        with torch.no_grad():
            dummy_input = torch.zeros(1, *input_size)  # Batch size 1
            features = self._forward_features(dummy_input)
            return features.numel()

    def _forward_features(self, x):
        """
        Forward pass through the feature extractor (without the fully connected layer).
        """
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

    def forward(self, x):
        x = self._forward_features(x)
        x = self.avgpool(x)
        x = self.fc(x)
        return x


In [8]:
def resnet18(input_size=(3, 224, 224), num_classes=4):
    """
    Constructs a ResNet-18 model.
    Args:
        input_size: Tuple representing the input size (C, H, W).
        num_classes: Number of output classes.
    """
    return ResNet(block, [2, 2, 2, 2], num_classes=num_classes, input_size=input_size)